# Day 09 — Custom Metrics

**Module 2 · The Metric Toolkit**

DeepEval provides many built-in metrics, but real applications often need
application-specific evaluation criteria.

Today we learn how to create a custom metric.

We will build two types:

1. **Deterministic metric** — no LLM required.
2. **LLM-based metric** — use an LLM judge for subjective evaluation.

> **Core idea:** If you can define how quality should be measured, you can turn that definition into a DeepEval metric.

## 1. Setup

We will use OpenAI as our evaluation judge.

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import OpenAIModel
from deepeval.evaluate import AsyncConfig

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

async_config = AsyncConfig(
    max_concurrent=2
)

print("Judge:", judge.get_model_name())
print("Max concurrent:", async_config.max_concurrent)

Judge: gpt-4.1-mini
Max concurrent: 2


## 2. Deterministic Custom Metric

Suppose our application should keep answers short.

This is a rule that does not require an LLM:

```text
Count words → calculate score → PASS / FAIL

In [1]:
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase


class MaxWordsMetric(BaseMetric):
    def __init__(self, max_words: int = 40, threshold: float = 0.5):
        self.max_words = max_words
        self.threshold = threshold
        super().__init__()

    def measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        word_count = len(test_case.actual_output.split())

        # Simple logic: Pass (1.0) or Fail (0.0)
        if word_count <= self.max_words:
            self.score = 1.0
            self.reason = f"Passed: {word_count} words (limit is {self.max_words})."
        else:
            self.score = 0.0
            self.reason = f"Failed: Exceeded limit with {word_count} words (limit is {self.max_words})."

        return self.score

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        return self.measure(test_case, *args, **kwargs)

## 3. Test the Custom Metric

Our metric follows the same basic DeepEval pattern:

```text
Test Case
    ↓
measure()
    ↓
score + reason
    ↓
threshold
    ↓
PASS / FAIL

In [2]:
short_case = LLMTestCase(
    input="What is RAG?",
    actual_output="RAG retrieves relevant documents and provides them to an LLM."
)

word_metric = MaxWordsMetric(
    max_words=15,
    threshold=0.5,
)

score = word_metric.measure(short_case)

print("Score:", score)
print("Reason:", word_metric.reason)
print("Success:", word_metric.is_successful())

Score: 1.0
Reason: Passed: 10 words (limit is 15).
Success: True


## 4. LLM-Based Custom Metric

Some qualities cannot be reliably measured with simple rules.

Examples:

- Professional tone
- Empathy
- Clarity
- Brand voice
- Whether an answer directly addresses a question

For these cases, we can create a custom metric that asks an LLM judge to evaluate the output.

In [4]:
class DirectAnswerMetric(BaseMetric):
    def __init__(self, model, threshold=0.5):
        self.model = model
        self.threshold = threshold
        super().__init__()

    def measure(self, test_case, *args, **kwargs):
        prompt = f"""
Determine whether the answer directly addresses the question.

Question:
{test_case.input}

Answer:
{test_case.actual_output}

Respond with exactly one word:
YES or NO
"""

        verdict, _ = self.model.generate(prompt)

        verdict = str(verdict).strip().upper()

        self.score = 1.0 if verdict == "YES" else 0.0
        self.reason = f"Judge verdict: {verdict}"

        return self.score

    async def a_measure(self, test_case, *args, **kwargs):
        return self.measure(test_case, *args, **kwargs)

## 5. Evaluate Real Test Cases

Now we will use our custom LLM-based metric just like any built-in DeepEval metric.

In [5]:
from deepeval import evaluate

cases = [
    LLMTestCase(
        input="What's the refund policy?",
        actual_output="Refunds are available within 30 days of purchase.",
    ),
    LLMTestCase(
        input="What's the refund policy?",
        actual_output="I really like trains and railway stations.",
    ),
]

direct_answer = DirectAnswerMetric(
    model=judge,
    threshold=0.5,
)

results = evaluate(
    test_cases=cases,
    metrics=[direct_answer],
    async_config=async_config,
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(f"\n{result.name}")
    print(f"Score:   {metric_result.score:.2f}")
    print(f"Success: {metric_result.success}")
    print(f"Reason:  {metric_result.reason}")

✨ You're running DeepEval's latest Base Metric Metric! (using None, strict=False, async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What's the refund policy?                                                              │
│  │     Actual Output:    I really like trains and railway stations.                                             │
│  └── Metrics                                                                                                    │
│          Status    ┃ Metric                ┃ Score       ┃ Threshold         ┃ Reason                           │
│      ━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│           FAIL     │ Base Metric           │ 0.00        │ 0.50              │ Judge verdict: NO                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric               ┃ Average Score           ┃ Pass Rate                                        ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Base Metric          │ 0.50                    │ 50.00% | passed=1 | failed=1                     │ 2          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=801425;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.41s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


test_case_0
Score:   1.00
Success: True
Reason:  Judge verdict: YES

test_case_1
Score:   0.00
Success: False
Reason:  Judge verdict: NO


## 6. The Custom Metric Contract

A custom DeepEval metric needs to provide:

- `threshold` — defines the pass/fail boundary.
- `measure()` — calculates the score.
- `a_measure()` — asynchronous version used during evaluation.
- `self.score` — stores the result.
- `self.reason` — explains the result.

DeepEval then handles the evaluation machinery around the metric.

## 7. Choosing the Right Approach

Use the simplest evaluation method that correctly measures the requirement.

| Requirement | Best starting point |
|---|---|
| Word count | Deterministic custom metric |
| JSON schema | Deterministic validation |
| Exact match | Deterministic comparison |
| Professional tone | G-Eval |
| Empathy | G-Eval |
| Application-specific logic | Custom metric |
| Complex judge behavior | LLM-based custom metric |

> **Rule:** Don't use an LLM judge when a deterministic rule can measure the requirement reliably.

# Day 09 — Key Takeaways

- DeepEval allows us to create application-specific metrics.
- Deterministic metrics are cheap, fast, and reproducible.
- LLM-based custom metrics are useful for subjective criteria.
- A custom metric produces a **score + reason** and can use a threshold for PASS / FAIL.
- Start with G-Eval when possible; move to a custom metric when you need more control.

### Evaluation hierarchy

```text
Built-in Metric
      ↓
    G-Eval
      ↓
Custom Metric
      ↓
Full Evaluation System